In [18]:
import os
import requests
from datetime import date, timedelta

url = "https://archive-api.open-meteo.com/v1/archive"

In [3]:
DAILY_VARIABLES = [
    "temperature_2m_max",       # Maximum Temperature (2 m)
    "temperature_2m_min",       # Minimum Temperature (2 m)
    "precipitation_sum",        # Precipitation Sum
    "windspeed_10m_max",        # Maximum Wind Speed (10 m)
    "relativehumidity_2m_mean", # Mean Relative Humidity (2 m) -- Additional Daily Variable
]

In [16]:
params = {
    "latitude": 33.7490,
    "longitude": -84.3880,
    "start_date": "2024-01-01",
    "end_date": "2024-01-31",
    "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
    "timezone": "America/New_York",
}

In [ ]:
requests.get(url, params=params, timeout=10)

<Response [200]>

In [6]:
LOOKBACK_DAYS = int(os.getenv("LOOKBACK_DAYS", 30))

In [ ]:
def get_date_range() -> tuple[str, str]:
    """
    Return (start_date, end_date) as ISO strings for the lookback window.

    The historical API (archive-api.open-meteo.com) lags by ~5 days.
    Using a 5-day lag on end_date ensures data is always available.
    """
    end = date.today() - timedelta(days=5)
    start = end - timedelta(days=LOOKBACK_DAYS - 1)
    return start.isoformat(), end.isoformat()

In [ ]:
get_date_range()

('2026-02-02', '2026-03-03')

In [12]:
CITIES = [
    {"name": "New York",     "latitude": 40.71280,  "longitude": -74.00600, "timezone": "America/New_York"},
    {"name": "Los Angeles",  "latitude": 34.05220,  "longitude": -118.24370, "timezone": "America/Los_Angeles"},
    {"name": "Chicago",      "latitude": 41.85003,  "longitude": -87.65005, "timezone": "America/Chicago"},
    {"name": "Houston",      "latitude": 29.76328,  "longitude": -95.36327, "timezone": "America/Chicago"},
    {"name": "Atlanta",      "latitude": 33.74900,  "longitude": -84.38798, "timezone": "America/New_York"},
    {"name": "London",       "latitude": 51.50853,  "longitude": -0.12574,  "timezone": "Europe/London"},
    {"name": "Paris",        "latitude": 48.85341,  "longitude":  2.34880,  "timezone": "Europe/Paris"},
    {"name": "Tokyo",        "latitude": 35.68950,  "longitude": 139.69171, "timezone": "Asia/Tokyo"},
    {"name": "Sydney",       "latitude": -33.86785, "longitude": 151.20732, "timezone": "Australia/Sydney"},
    {"name": "Nairobi",      "latitude": -1.28333,  "longitude":  36.81667, "timezone": "Africa/Nairobi"},
]

In [ ]:
GCS_BUCKET_NAME = os.getenv("GCS_BUCKET_NAME")

'de-zoomcamp-489713-raw-weather-data'

In [15]:
def gcs_raw_key(city_name: str, run_date: str) -> str:
    """
    GCS object key for a raw API response.
    Pattern: raw/{city_name}/{run_date}.json
    Example: raw/New York/2024-03-10.json
    """
    safe_name = city_name.replace(" ", "_").lower()
    return f"raw/{safe_name}/{run_date}.json"